In [ ]:
# ==============================================================================
# ETL PIPELINE: SILVER TO GOLD LAYER
# Analytics Aggregation & Business KPI Calculations
# ==============================================================================

import os
import sys
import logging
from datetime import datetime
from pathlib import Path

from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, sum as spark_sum, count as spark_count, avg, countDistinct,
    max as spark_max, min as spark_min, round, dense_rank,
    current_timestamp, lit
)
from delta.tables import DeltaTable

# ── Initialize Spark Session ────────────────────────────────────────────────
spark = SparkSession.builder \
    .appName("EcommercePipeline-SilverToGold") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# ── Setup Logging ──────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
logger.info("="*80)
logger.info("STARTING: Silver to Gold ETL Pipeline")
logger.info("="*80)

# ── Define Local Paths (Relative to project root) ─────────────────────────
BASE_PATH = Path("./data")
SILVER_PATH = str(BASE_PATH / "silver" / "ecommerce_sales")
GOLD_PATH = str(BASE_PATH / "gold")

# Create directories if they don't exist
Path(GOLD_PATH).mkdir(parents=True, exist_ok=True)

logger.info(f"Silver path: {SILVER_PATH}")
logger.info(f"Gold path: {GOLD_PATH}")

In [ ]:
# ── Step 6: Pipeline Execution Summary ─────────────────────────────────────

logger.info("\n" + "="*80)
logger.info("ETL PIPELINE EXECUTION SUMMARY")
logger.info("="*80)

tables_created = [
    ("monthly_sales_by_category_region", gold_monthly.count()),
    ("payment_analysis", gold_payment.count()),
    ("top_products", gold_products.count())
]

logger.info("\n✓ Gold Layer Tables Created:")
for table_name, record_count in tables_created:
    logger.info(f"  - {table_name}: {record_count:,} records")

logger.info("\n📁 Output Locations:")
logger.info(f"  Monthly Sales: {gold_monthly_path}")
logger.info(f"  Payment Analysis: {gold_payment_path}")
logger.info(f"  Product Performance: {gold_products_path}")

logger.info("\n" + "="*80)
logger.info("✓ SILVER TO GOLD ETL PIPELINE COMPLETED SUCCESSFULLY")
logger.info("="*80)
logger.info("\nNext Steps:")
logger.info("1. Connect Gold tables to BI dashboards (Power BI, Tableau, Metabase)")
logger.info("2. Schedule notebooks to run on a defined cadence")
logger.info("3. Set up data quality monitoring and alerts")
logger.info("4. Document business logic and data lineage")

## Step 6: Pipeline Execution Summary

Generate final summary metrics and export statistics for pipeline monitoring.

In [ ]:
# ── Step 5: Business KPI Calculations ──────────────────────────────────────

logger.info("\n" + "="*80)
logger.info("BUSINESS KPI CALCULATIONS")
logger.info("="*80)

# Overall metrics
total_revenue = float(df_silver.agg(spark_sum("sales")).collect()[0][0])
total_profit = float(df_silver.agg(spark_sum("profit")).collect()[0][0])
total_orders = df_silver.count()
unique_customers = df_silver.select(countDistinct("customer_name")).collect()[0][0]
avg_order_value = float(df_silver.agg(avg("sales")).collect()[0][0])
profit_margin = (total_profit / total_revenue * 100) if total_revenue > 0 else 0

# Regional breakdown
regional_stats = df_silver.groupBy("region").agg(
    spark_sum("sales").alias("revenue"),
    spark_sum("profit").alias("profit"),
    spark_count("order_id").alias("orders")
).collect()

# Category breakdown
category_stats = df_silver.groupBy("category").agg(
    spark_sum("sales").alias("revenue"),
    spark_sum("profit").alias("profit"),
    spark_count("order_id").alias("orders")
).orderBy(col("revenue").desc()).collect()

logger.info("\n📊 OVERALL BUSINESS KPIs")
logger.info(f"  Total Revenue: ₹{total_revenue:,.2f}")
logger.info(f"  Total Profit: ₹{total_profit:,.2f}")
logger.info(f"  Profit Margin: {profit_margin:.2f}%")
logger.info(f"  Total Orders: {total_orders:,}")
logger.info(f"  Unique Customers: {unique_customers:,}")
logger.info(f"  Average Order Value: ₹{avg_order_value:,.2f}")

logger.info("\n📍 REVENUE BY REGION")
for stat in regional_stats:
    region, revenue, profit, orders = stat
    logger.info(f"  {region}: ₹{revenue:,.2f} ({orders:,} orders)")

logger.info("\n📦 TOP CATEGORIES BY REVENUE")
for stat in category_stats[:5]:
    category, revenue, profit, orders = stat
    logger.info(f"  {category}: ₹{revenue:,.2f}")

## Step 5: Calculate Business KPIs

Compute key performance indicators for executive dashboards and business reporting.

In [ ]:
# ── Step 4: Gold Table - Top Products Performance ──────────────────────────

logger.info("\n" + "="*80)
logger.info("CREATING GOLD TABLE: PRODUCT PERFORMANCE")
logger.info("="*80)

# Aggregate by product and category
gold_products = df_silver.groupBy(
    "category", "sub_category", "product_name"
).agg(
    spark_sum("quantity").alias("total_units_sold"),
    spark_sum("sales").alias("total_revenue"),
    spark_sum("profit").alias("total_profit"),
    spark_count("order_id").alias("order_count"),
    avg("profit_margin_pct").alias("avg_profit_margin_pct"),
    avg("sales").alias("avg_order_value")
).withColumn(
    "revenue_rank",
    dense_rank().over(Window.partitionBy("category").orderBy(col("total_revenue").desc()))
).withColumn(
    "_gold_load_timestamp", current_timestamp()
)

# Save to Gold layer
gold_products_path = str(Path(GOLD_PATH) / "top_products")

try:
    gold_products.coalesce(1).write \
        .format("parquet") \
        .mode("overwrite") \
        .save(gold_products_path)
    
    logger.info(f"✓ Gold Table created: top_products")
    logger.info(f"  Records: {gold_products.count():,}")
    logger.info(f"  Location: {gold_products_path}")
except Exception as e:
    logger.error(f"✗ Error creating product performance table: {str(e)}")
    raise

logger.info("\nTop 10 products by revenue:")
gold_products.orderBy(col("total_revenue").desc()).show(10)

## Step 4: Gold Table 3 - Top Products Performance

Create a product performance analysis table to identify top-performing products and categories.

In [ ]:
# ── Step 3: Gold Table - Payment Analysis ─────────────────────────────────

logger.info("\n" + "="*80)
logger.info("CREATING GOLD TABLE: PAYMENT ANALYSIS")
logger.info("="*80)

# Aggregate by payment method and status
gold_payment = df_silver.groupBy(
    "payment_mode", "payment_status", "region"
).agg(
    spark_count("order_id").alias("transaction_count"),
    spark_sum("sales").alias("total_transaction_value"),
    avg("sales").alias("avg_transaction_value"),
    spark_sum("profit").alias("total_profit")
).withColumn(
    "profit_margin_pct",
    round(col("total_profit") / col("total_transaction_value") * 100, 2)
).withColumn(
    "_gold_load_timestamp", current_timestamp()
)

# Save to Gold layer
gold_payment_path = str(Path(GOLD_PATH) / "payment_analysis")

try:
    gold_payment.coalesce(1).write \
        .format("parquet") \
        .mode("overwrite") \
        .save(gold_payment_path)
    
    logger.info(f"✓ Gold Table created: payment_analysis")
    logger.info(f"  Records: {gold_payment.count():,}")
    logger.info(f"  Location: {gold_payment_path}")
except Exception as e:
    logger.error(f"✗ Error creating payment analysis table: {str(e)}")
    raise

logger.info("\nPayment distribution:")
gold_payment.show(10)

## Step 3: Gold Table 2 - Payment Analysis

Create a payment method analysis table for understanding transaction patterns across different payment modes and statuses.

In [ ]:
# ── Step 2: Gold Table - Monthly Sales by Category & Region ───────────────

logger.info("\n" + "="*80)
logger.info("CREATING GOLD TABLE: MONTHLY SALES AGGREGATION")
logger.info("="*80)

# Aggregate by time period, category, and region
gold_monthly = df_silver.groupBy(
    "order_year", "order_month", "order_quarter",
    "category", "region"
).agg(
    spark_sum("sales").alias("total_sales"),
    spark_sum("net_sales").alias("total_net_sales"),
    spark_sum("profit").alias("total_profit"),
    spark_count("order_id").alias("total_orders"),
    avg("discount").alias("avg_discount_pct"),
    avg("profit_margin_pct").alias("avg_profit_margin_pct"),
    countDistinct("customer_name").alias("unique_customers"),
    avg("sales").alias("avg_order_value")
).withColumn(
    "_gold_load_timestamp", current_timestamp()
).withColumn(
    "profit_to_sales_ratio",
    round(col("total_profit") / col("total_sales"), 4)
)

# Save to Gold layer
gold_monthly_path = str(Path(GOLD_PATH) / "monthly_sales_by_category_region")

try:
    gold_monthly.coalesce(1).write \
        .format("parquet") \
        .mode("overwrite") \
        .save(gold_monthly_path)
    
    logger.info(f"✓ Gold Table created: monthly_sales_by_category_region")
    logger.info(f"  Records: {gold_monthly.count():,}")
    logger.info(f"  Location: {gold_monthly_path}")
except Exception as e:
    logger.error(f"✗ Error creating monthly sales table: {str(e)}")
    raise

logger.info("\nSample of monthly sales data:")
gold_monthly.orderBy(col("total_sales").desc()).show(5)

## Step 2: Gold Table 1 - Monthly Sales by Category & Region

Aggregate sales data by temporal dimensions and product categories to create a comprehensive monthly sales fact table.

In [ ]:
# ── Step 1: Load Data from Silver Layer ────────────────────────────────────

logger.info("\n" + "="*80)
logger.info("LOADING DATA FROM SILVER LAYER")
logger.info("="*80)

try:
    df_silver = spark.read.format("delta").load(SILVER_PATH)
    silver_count = df_silver.count()
    logger.info(f"✓ Successfully loaded {silver_count:,} records from Silver layer")
    logger.info(f"  Location: {SILVER_PATH}")
    logger.info("\nSilver layer schema:")
    df_silver.printSchema()
except Exception as e:
    logger.error(f"✗ Error loading Silver layer: {str(e)}")
    raise

## Step 1: Load Data from Silver Layer

Load the cleaned and transformed data from the Silver layer for aggregation.

# Silver → Gold Layer: Analytics & Aggregation

This notebook demonstrates the final stage of the ETL pipeline where cleaned data is aggregated into analytics-ready business tables and KPI calculations.

**Medallion Architecture:** Bronze (raw) → Silver (cleaned) → Gold (aggregated)

**Processing Steps:**
- Read cleaned data from Silver layer
- Build analytics-ready aggregated tables using groupBy and window functions
- Calculate business KPIs (revenue, profit, customer metrics)
- Create dimension and fact tables following star schema principles
- Implement incremental processing for scalability
- Export datasets for dashboards and reporting

**Output Tables:**
- Monthly sales by category/region (fact table)
- Payment method analysis
- Top performing products
- Customer acquisition metrics